In [27]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalETL")
    # === MEMORY ALLOCATION ===
    .config("spark.driver.memory", "10g")        # main process (driver)
    .config("spark.executor.memory", "10g")      # worker processes (same JVM locally)
    .config("spark.driver.maxResultSize", "2g")  # prevent result collection errors
    
    # === PARALLELISM ===
    .config("spark.driver.cores", "6")           # use 6 out of 8 cores
    .config("spark.executor.cores", "6")
    .config("spark.default.parallelism", "12")   # usually ~2x num_cores for local
    
    # === PERFORMANCE TUNING ===
    .config("spark.sql.shuffle.partitions", "24")  # parallelize shuffles (joins/groupBy)
    .config("spark.memory.fraction", "0.8")        # 80% of JVM heap for Spark execution
    .config("spark.memory.storageFraction", "0.4") # 40% of that for caching
    .config("spark.sql.files.maxPartitionBytes", "128MB")  # ideal partition size for I/O
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # fast Pandas conversion
    .config("spark.local.dir", "/tmp/spark-temp")  # local disk space for shuffle spill
    
    # === OPTIONAL LOGGING + CLEANUP ===
    .config("spark.sql.broadcastTimeout", "600")  # allow large table broadcasts
    .config("spark.cleaner.periodicGC.interval", "5min")  # reduce memory leaks
    
    .getOrCreate()
)


In [28]:

from pyspark.sql import Window
from pyspark.sql import functions as F

fraud_ranks = spark.read.csv("../data/curated/merchant_fraud_rankings.csv", header=True, inferSchema=True)
growth_ranks = spark.read.csv("../data/curated/merchant_growth_rankings.csv", header=True, inferSchema = True)
merchant_transactions = spark.read.parquet("../data/curated/merchant_transactions")


In [29]:
fraud_ranks.show(10)

+------------+-----+------------------+-------------------+------------------+-------------------+-------------------+--------------------+--------------------+
|merchant_abn|n_txn|        sum_amount|             mean_p|               EFL|               EFLR|               eb_p|           se_mean_p|       merchant_name|
+------------+-----+------------------+-------------------+------------------+-------------------+-------------------+--------------------+--------------------+
| 24015173965| 7849|         1232293.0|0.15507555227517678|191098.51753983443|0.15507555227517678|0.15502733775150318|0.004085765547267716|      Lectus Limited|
| 36325756075|  116|29600.849351309553|0.15389524000588256| 4779.047517689569|0.16144967534447258| 0.1541998066885062|0.033503923484784016|       Amet Diam Ltd|
| 49068909356|  136| 38069.74966543718|0.15094636171618783| 5627.158957794549|0.14781181928557155|0.15355965137238833| 0.03069795423352795|    Euismod Est Inc.|
| 50446282783|   68|10748.98442553

In [30]:
growth_ranks.show(10)

+------------+------+------+------+-------+------------------+
|merchant_abn|rank_1|rank_3|rank_6|rank_12|          ave_rank|
+------------+------+------+------+-------+------------------+
| 49505931725| 102.8| 368.2| 218.4|   59.0|             187.1|
| 49322182190|  69.0| 433.2|  NULL|   69.4|190.53333333333333|
| 88699453206| 635.2|  64.8|  67.8|   91.2|            214.75|
| 79417999332|  33.8| 239.8| 273.0|  326.8|218.35000000000002|
| 38918664617|  NULL| 229.4|  NULL|   NULL|             229.4|
| 78760357380| 327.8| 129.0| 132.2|  346.4|            233.85|
| 80518954462|  55.6|  48.8| 557.6|  316.6|            244.65|
| 93558142492| 108.2|  87.6| 579.0|  285.4|265.04999999999995|
| 66228393506| 129.2|  79.4| 372.8|  506.2|             271.9|
| 72472909171|  83.6| 363.8| 265.0|  403.2|             278.9|
+------------+------+------+------+-------+------------------+
only showing top 10 rows



In [31]:
merchant_transactions.show(10)
merchant_transactions_agg = merchant_transactions.groupBy('merchant_abn', 'segment', 'business', 'take_rate').agg(
    F.sum('dollar_value').alias('total_revenue'),
    F.min("order_datetime").alias("first_order_date"),
    F.max("order_datetime").alias("last_order_date")

    )

joined_rankings = merchant_transactions_agg.join(
    fraud_ranks, on='merchant_abn', how='left').join(
        growth_ranks, on='merchant_abn', how='left')

joined_rankings = joined_rankings.withColumn(
    "date_range_months",
    F.ceil(F.months_between(F.col("last_order_date"), F.col("first_order_date")))
).drop("first_order_date", "last_order_date")

joined_rankings = joined_rankings.withColumn(
    "avg_monthly_takings",
    F.col('take_rate') / 100 * F.col("total_revenue") / F.col("date_range_months")
)

joined_rankings.orderBy('avg_monthly_takings', ascending=False)

window = Window.orderBy(F.desc("avg_monthly_takings"))

joined_rankings = joined_rankings.withColumn("Takings Rank", F.row_number().over(window))

joined_rankings = joined_rankings.withColumn(
    "loss_potential",
    F.col("avg_monthly_takings") * (F.col("eb_p"))
)
window = Window.orderBy(F.desc("loss_potential"))

joined_rankings = joined_rankings.withColumn("Loss Potential Ranking", F.row_number().over(window))

window = Window.orderBy(F.asc("ave_rank"))

joined_rankings = joined_rankings.withColumn("Growth Potential Ranking", F.row_number().over(window))

joined_rankings = joined_rankings.withColumn(
    "composite_ranking",
    (F.col("Takings Rank") + F.col("Loss Potential Ranking") + F.col("Growth Potential Ranking"))/3
)

window = Window.orderBy(F.asc("composite_ranking"))

joined_rankings = joined_rankings.withColumn("Final Ranking", F.row_number().over(window))

joined_rankings= joined_rankings.select(
    "merchant_abn", "merchant_name", "segment", "Takings Rank", 
    "Loss Potential Ranking", "Growth Potential Ranking", "Final Ranking")

joined_rankings.show(100, truncate=False)


+------------+-------+------------------+--------------+--------------------+--------------------+--------+---------+--------------------+
|merchant_abn|user_id|      dollar_value|order_datetime|            business|            biz_tags|rev_band|take_rate|             segment|
+------------+-------+------------------+--------------+--------------------+--------------------+--------+---------+--------------------+
| 34096466752|      3| 301.5793450525113|    2021-08-20|     Nullam Enim Ltd|computers, comput...|       b|     3.22|Technology & Prof...|
| 70501974849|   5855|133.88189838137183|    2022-03-26|Facilisis Lorem T...|computers, comput...|       b|     3.30|Technology & Prof...|
| 70501974849|  18482| 68.75486276223054|    2021-08-20|Facilisis Lorem T...|computers, comput...|       b|     3.30|Technology & Prof...|
| 77338620996|   5869|  441.130116377912|    2022-03-26| Fames Ac Turpis LLC|computers, comput...|       b|     3.59|Technology & Prof...|
| 68216911708|     14| 32.9

25/10/09 20:05:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

+------------+---------------------------------+----------------------------------+------------+----------------------+------------------------+-------------+
|merchant_abn|merchant_name                    |segment                           |Takings Rank|Loss Potential Ranking|Growth Potential Ranking|Final Ranking|
+------------+---------------------------------+----------------------------------+------------+----------------------+------------------------+-------------+
|49322182190 |Gravida Mauris Incorporated      |Fashion, Jewelry & Personal Goods |10          |11                    |114                     |1            |
|96680767841 |Ornare Limited                   |Lifestyle, Health & Recreation    |4           |5                     |126                     |2            |
|45629217853 |Lacus Consulting                 |Fashion, Jewelry & Personal Goods |3           |3                     |137                     |3            |
|79417999332 |Phasellus At Company            

25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

In [32]:
joined_rankings.write.csv("../data/final_ranks/final_merchant_rankings.csv", header=True, mode="overwrite")

25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

In [33]:
h_g_l_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Home, Garden & Living').orderBy('Final Ranking')

h_g_l_segment_rank.show(10, truncate=False)

25/10/09 20:05:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

+------------+------------------------------+---------------------+------------+----------------------+------------------------+-------------+
|merchant_abn|merchant_name                 |segment              |Takings Rank|Loss Potential Ranking|Growth Potential Ranking|Final Ranking|
+------------+------------------------------+---------------------+------------+----------------------+------------------------+-------------+
|89726005175 |Est Nunc Consulting           |Home, Garden & Living|8           |7                     |132                     |5            |
|76767266140 |Phasellus At Limited          |Home, Garden & Living|18          |18                    |129                     |9            |
|38090089066 |Interdum Feugiat Sed Inc.     |Home, Garden & Living|38          |37                    |130                     |20           |
|21772962346 |Purus Gravida Sagittis Ltd    |Home, Garden & Living|37          |36                    |168                     |31           |

In [34]:
l_h_r_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Lifestyle, Health & Recreation').orderBy('Final Ranking')

l_h_r_segment_rank.show(10, truncate=False)

25/10/09 20:05:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

+------------+-------------------------------+------------------------------+------------+----------------------+------------------------+-------------+
|merchant_abn|merchant_name                  |segment                       |Takings Rank|Loss Potential Ranking|Growth Potential Ranking|Final Ranking|
+------------+-------------------------------+------------------------------+------------+----------------------+------------------------+-------------+
|96680767841 |Ornare Limited                 |Lifestyle, Health & Recreation|4           |5                     |126                     |2            |
|48534649627 |Dignissim Maecenas Foundation  |Lifestyle, Health & Recreation|12          |13                    |123                     |6            |
|88699453206 |Sed Nec Inc.                   |Lifestyle, Health & Recreation|67          |67                    |115                     |33           |
|46804135891 |Suspendisse Dui Corporation    |Lifestyle, Health & Recreation|63   

25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

In [35]:
t_p_s_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Technology & Professional Services').orderBy('Final Ranking')

t_p_s_segment_rank.show(10, truncate=False)

25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

+------------+-------------------------------+----------------------------------+------------+----------------------+------------------------+-------------+
|merchant_abn|merchant_name                  |segment                           |Takings Rank|Loss Potential Ranking|Growth Potential Ranking|Final Ranking|
+------------+-------------------------------+----------------------------------+------------+----------------------+------------------------+-------------+
|57757792876 |Pretium Et LLC                 |Technology & Professional Services|29          |29                    |135                     |16           |
|45433476494 |Adipiscing Elit Foundation     |Technology & Professional Services|20          |22                    |160                     |18           |
|80518954462 |Neque Sed Dictum Incorporated  |Technology & Professional Services|44          |44                    |119                     |22           |
|82368304209 |Nec Incorporated               |Technology &

In [38]:
a_m_e_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Arts, Media & Entertainment').orderBy('Final Ranking')

a_m_e_segment_rank.show(10, truncate=False)

25/10/09 20:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

+------------+---------------------------------+---------------------------+------------+----------------------+------------------------+-------------+
|merchant_abn|merchant_name                    |segment                    |Takings Rank|Loss Potential Ranking|Growth Potential Ranking|Final Ranking|
+------------+---------------------------------+---------------------------+------------+----------------------+------------------------+-------------+
|72472909171 |Nullam Consulting                |Arts, Media & Entertainment|14          |16                    |122                     |7            |
|49505931725 |Suspendisse Ac Associates        |Arts, Media & Entertainment|31          |31                    |113                     |12           |
|98166254020 |Magna Sed Industries             |Arts, Media & Entertainment|17          |19                    |139                     |13           |
|21439773999 |Mauris Non Institute             |Arts, Media & Entertainment|5           

25/10/09 20:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

In [37]:
f_j_p_segment_rank = joined_rankings.filter(joined_rankings['segment'] == 'Fashion, Jewelry & Personal Goods').orderBy('Final Ranking')

f_j_p_segment_rank.show(10, truncate=False)

25/10/09 20:05:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 20:05:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/10/09 2

+------------+-----------------------------+---------------------------------+------------+----------------------+------------------------+-------------+
|merchant_abn|merchant_name                |segment                          |Takings Rank|Loss Potential Ranking|Growth Potential Ranking|Final Ranking|
+------------+-----------------------------+---------------------------------+------------+----------------------+------------------------+-------------+
|49322182190 |Gravida Mauris Incorporated  |Fashion, Jewelry & Personal Goods|10          |11                    |114                     |1            |
|45629217853 |Lacus Consulting             |Fashion, Jewelry & Personal Goods|3           |3                     |137                     |3            |
|79417999332 |Phasellus At Company         |Fashion, Jewelry & Personal Goods|15          |15                    |116                     |4            |
|86578477987 |Leo In Consulting            |Fashion, Jewelry & Personal Good